In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Créer le schema s'il n'existe pas encore
session.sql("USE DATABASE HOUSE_PRICE_DB").collect()
session.sql("CREATE SCHEMA IF NOT EXISTS ML_SCHEMA").collect()
session.sql("USE SCHEMA ML_SCHEMA").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()

print("Session active :", session.get_current_database(), "/", session.get_current_schema())

In [ ]:
# Chargement des données depuis S3
session.sql("""
    CREATE OR REPLACE TABLE HOUSE_PRICES (
        price            FLOAT,
        area             FLOAT,
        bedrooms         INTEGER,
        bathrooms        INTEGER,
        stories          INTEGER,
        mainroad         VARCHAR(5),
        guestroom        VARCHAR(5),
        basement         VARCHAR(5),
        hotwaterheating  VARCHAR(5),
        airconditioning  VARCHAR(5),
        parking          INTEGER,
        prefarea         VARCHAR(5),
        furnishingstatus VARCHAR(20)
    )
""").collect()
print("Table créée ")

session.sql("""
    CREATE OR REPLACE STAGE house_price_stage
    URL = 's3://logbrain-datalake/datasets/house_price/'
    FILE_FORMAT = (TYPE='CSV' FIELD_OPTIONALLY_ENCLOSED_BY='"' SKIP_HEADER=1)
""").collect()
print("Stage S3 créé ")

session.sql("""
    COPY INTO HOUSE_PRICES
    FROM @house_price_stage
    ON_ERROR = 'CONTINUE'
""").collect()
print("Données chargées ")

count = session.table("HOUSE_PRICES").count()
print(f"Nombre de lignes : {count}")


In [ ]:
# Vérifier les fichiers disponibles dans le stage S3
result = session.sql("LIST @house_price_stage").collect()
for row in result:
    print(row)

In [ ]:
# Étape 1 - Recréer le stage en JSON
session.sql("""
    CREATE OR REPLACE STAGE house_price_stage
    URL = 's3://logbrain-datalake/datasets/house_price/'
    FILE_FORMAT = (TYPE = 'JSON' STRIP_OUTER_ARRAY = TRUE)
""").collect()
print("Stage JSON créé ")

# Étape 2 - Table de staging brute pour lire le JSON
session.sql("""
    CREATE OR REPLACE TABLE HOUSE_PRICES_RAW (raw VARIANT)
""").collect()

session.sql("""
    COPY INTO HOUSE_PRICES_RAW
    FROM @house_price_stage/Housing_Price_Data.json
    FILE_FORMAT = (TYPE = 'JSON' STRIP_OUTER_ARRAY = TRUE)
    ON_ERROR = 'CONTINUE'
""").collect()

count_raw = session.sql("SELECT COUNT(*) FROM HOUSE_PRICES_RAW").collect()[0][0]
print(f"Lignes brutes chargées : {count_raw}")

# Étape 3 - Voir la structure du JSON
session.sql("SELECT raw FROM HOUSE_PRICES_RAW LIMIT 2").show()


In [ ]:
df_snow = session.table("HOUSE_PRICES")
print(f"Nombre de lignes : {df_snow.count()}")
df_snow.show(5)

df = df_snow.to_pandas()
print("\nColonnes disponibles :", df.columns.tolist())
print("\nTypes :")
print(df.dtypes)

In [ ]:
print("=== Valeurs nulles ===")
print(df.isnull().sum())

print("\n=== Valeurs uniques des colonnes catégorielles ===")
cat_cols = ["MAINROAD", "GUESTROOM", "BASEMENT", "HOTWATERHEATING",
            "AIRCONDITIONING", "PREFAREA", "FURNISHINGSTATUS"]
for col in cat_cols:
    print(f"  {col}: {df[col].unique()}")


In [ ]:
print("Lignes dans la table Snowflake :", session.table("HOUSE_PRICES").count())
print("Lignes dans df                  :", len(df))
print("Shape df                        :", df.shape)
print(df.head(3))

In [ ]:
# Vérifier HOUSE_PRICES_RAW d'abord
count_raw = session.sql("SELECT COUNT(*) FROM HOUSE_PRICES_RAW").collect()[0][0]
print(f"HOUSE_PRICES_RAW contient : {count_raw} lignes")

# Si 0, recharger depuis S3
if count_raw == 0:
    session.sql("""
        CREATE OR REPLACE FILE FORMAT json_format
        TYPE = 'JSON'
        STRIP_OUTER_ARRAY = TRUE
    """).collect()

    session.sql("""
        CREATE OR REPLACE STAGE house_price_stage
        URL = 's3://logbrain-datalake/datasets/house_price/'
        FILE_FORMAT = json_format
    """).collect()

    session.sql("TRUNCATE TABLE IF EXISTS HOUSE_PRICES_RAW").collect()
    session.sql("""
        CREATE OR REPLACE TABLE HOUSE_PRICES_RAW (raw VARIANT)
    """).collect()

    result = session.sql("""
        COPY INTO HOUSE_PRICES_RAW
        FROM @house_price_stage/Housing_Price_Data.json
        FILE_FORMAT = json_format
        ON_ERROR = 'CONTINUE'
    """).collect()
    print("Résultat COPY INTO :", result)
    count_raw = session.sql("SELECT COUNT(*) FROM HOUSE_PRICES_RAW").collect()[0][0]
    print(f"HOUSE_PRICES_RAW après rechargement : {count_raw} lignes")

# Recréer HOUSE_PRICES depuis RAW
session.sql("""
    CREATE OR REPLACE TABLE HOUSE_PRICES AS
    SELECT
        raw:price::FLOAT              AS price,
        raw:area::FLOAT               AS area,
        raw:bedrooms::INTEGER         AS bedrooms,
        raw:bathrooms::INTEGER        AS bathrooms,
        raw:stories::INTEGER          AS stories,
        raw:mainroad::VARCHAR         AS mainroad,
        raw:guestroom::VARCHAR        AS guestroom,
        raw:basement::VARCHAR         AS basement,
        raw:hotwaterheating::VARCHAR  AS hotwaterheating,
        raw:airconditioning::VARCHAR  AS airconditioning,
        raw:parking::INTEGER          AS parking,
        raw:prefarea::VARCHAR         AS prefarea,
        raw:furnishingstatus::VARCHAR AS furnishingstatus
    FROM HOUSE_PRICES_RAW
    WHERE raw:price IS NOT NULL
""").collect()

count_final = session.table("HOUSE_PRICES").count()
print(f"\nHOUSE_PRICES final : {count_final} lignes")

# Recharger dans pandas
df = session.table("HOUSE_PRICES").to_pandas()
print(f"df chargé : {len(df)} lignes")
print(df.head(3))


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df_prep = df.copy()

# 1. Encodage binaire : yes → 1, no → 0
binary_cols = ["MAINROAD", "GUESTROOM", "BASEMENT",
               "HOTWATERHEATING", "AIRCONDITIONING", "PREFAREA"]
for col in binary_cols:
    df_prep[col] = (df_prep[col].str.strip().str.lower() == "yes").astype(int)

# 2. Encodage ordinal : furnishingstatus
furnishing_map = {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}
df_prep["FURNISHINGSTATUS"] = (
    df_prep["FURNISHINGSTATUS"].str.strip().str.lower().map(furnishing_map)
)

print("=== Aperçu après encodage ===")
print(df_prep.head(5))

# 3. Séparation features / target
feature_cols = [
    "AREA", "BEDROOMS", "BATHROOMS", "STORIES",
    "MAINROAD", "GUESTROOM", "BASEMENT", "HOTWATERHEATING",
    "AIRCONDITIONING", "PARKING", "PREFAREA", "FURNISHINGSTATUS"
]
TARGET = "PRICE"

X = df_prep[feature_cols]
y = df_prep[TARGET]

# 4. Split train / test 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"\nTrain : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")

# 5. Normalisation
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols)

print("\nNormalisation OK")
print(X_train_scaled.describe().round(3))

In [ ]:
# CELLULE 5 — ENTRAÎNEMENT DES 6 MODÈLES

from sklearn.linear_model  import LinearRegression, Ridge, Lasso
from sklearn.ensemble      import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics       import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

def evaluate(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    rmse = np.sqrt(mean_squared_error(yte, pred))
    mae  = mean_absolute_error(yte, pred)
    r2   = r2_score(yte, pred)
    print(f"  [{name:<25}]  RMSE: {rmse:>12,.0f}  |  MAE: {mae:>12,.0f}  |  R²: {r2:.4f}")
    return {"name": name, "model": model, "pred": pred, "rmse": rmse, "mae": mae, "r2": r2}

models_to_train = {
    "LinearRegression"  : LinearRegression(),
    "Ridge (alpha=1)"   : Ridge(alpha=1.0),
    "Lasso (alpha=100)" : Lasso(alpha=100),
    "RandomForest"      : RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting"  : GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost"           : xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
}

print("=== Entraînement des modèles ===\n")
results = {}
for name, model in models_to_train.items():
    results[name] = evaluate(name, model, X_train_scaled, X_test_scaled, y_train, y_test)


In [ ]:
# CELLULE 6 — COMPARAISON VISUELLE DES PERFORMANCES

metrics_df = pd.DataFrame([
    {"Modèle": v["name"], "RMSE": v["rmse"], "MAE": v["mae"], "R²": v["r2"]}
    for v in results.values()
]).sort_values("R²", ascending=False).reset_index(drop=True)

print("=== Tableau récapitulatif (trié par R²) ===")
print(metrics_df.to_string(index=False))

# Graphique comparaison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Comparaison des modèles de base", fontsize=14, fontweight="bold")

for ax, (metric, label, color) in zip(axes, [
    ("RMSE", "↓ meilleur", "tomato"),
    ("MAE",  "↓ meilleur", "orange"),
    ("R²",   "↑ meilleur", "steelblue")
]):
    bars = ax.bar(metrics_df["Modèle"], metrics_df[metric], color=color, edgecolor="black")
    ax.set_title(f"{metric} ({label})", fontweight="bold")
    ax.tick_params(axis="x", rotation=40)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h * 1.01,
                f"{h:,.2f}", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.show()

# Prédictions vs réelles du meilleur modèle de base
best_base_name = metrics_df.iloc[0]["Modèle"]
best_base_res  = results[best_base_name]
print(f"\nMeilleur modèle de base : {best_base_name}")

plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_base_res["pred"], alpha=0.5, color="steelblue")
mn, mx = y_test.min(), y_test.max()
plt.plot([mn, mx], [mn, mx], "r--", lw=2, label="Prédiction parfaite")
plt.xlabel("Prix réels")
plt.ylabel("Prix prédits")
plt.title(f"{best_base_name} — Prédits vs Réels\nR²={best_base_res['r2']:.4f}")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# CELLULE 7 — OPTIMISATION DES HYPERPARAMÈTRES

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# GridSearch sur RandomForest
print("=== GridSearch — RandomForest ===")
rf_param_grid = {
    "n_estimators"     : [100, 200, 300],
    "max_depth"        : [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf" : [1, 2, 4],
    "max_features"     : ["sqrt", "log2"],
}
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_param_grid, cv=5, scoring="r2", n_jobs=-1, verbose=1
)
rf_grid.fit(X_train_scaled, y_train)
print(f"\nMeilleurs paramètres RF : {rf_grid.best_params_}")
print(f"Meilleur R² en CV       : {rf_grid.best_score_:.4f}")

# RandomSearch sur XGBoost
print("\n=== RandomizedSearch — XGBoost ===")
xgb_param_dist = {
    "n_estimators"    : [100, 200, 300, 500],
    "max_depth"       : [3, 5, 7, 10],
    "learning_rate"   : [0.01, 0.05, 0.1, 0.2],
    "subsample"       : [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha"       : [0, 0.1, 0.5, 1.0],
    "reg_lambda"      : [1, 2, 5, 10],
}
xgb_random = RandomizedSearchCV(
    xgb.XGBRegressor(random_state=42, verbosity=0),
    xgb_param_dist,
    n_iter=40, cv=5, scoring="r2", n_jobs=-1, random_state=42, verbose=1
)
xgb_random.fit(X_train_scaled, y_train)
print(f"\nMeilleurs paramètres XGB : {xgb_random.best_params_}")
print(f"Meilleur R² en CV        : {xgb_random.best_score_:.4f}")


In [ ]:
# CELLULE 8 — ÉVALUATION DES MODÈLES OPTIMISÉS

print("\n=== Évaluation sur le Test Set — Modèles optimisés ===\n")
rf_tuned_res  = evaluate("RandomForest (tuned)", rf_grid.best_estimator_,
                          X_train_scaled, X_test_scaled, y_train, y_test)
xgb_tuned_res = evaluate("XGBoost (tuned)",      xgb_random.best_estimator_,
                          X_train_scaled, X_test_scaled, y_train, y_test)

# Sélection du modèle final
if rf_tuned_res["r2"] >= xgb_tuned_res["r2"]:
    final_res   = rf_tuned_res
    final_model = rf_grid.best_estimator_
else:
    final_res   = xgb_tuned_res
    final_model = xgb_random.best_estimator_

print(f"\n>>> MODÈLE FINAL : {final_res['name']}")
print(f"    RMSE : {final_res['rmse']:,.0f}")
print(f"    MAE  : {final_res['mae']:,.0f}")
print(f"    R²   : {final_res['r2']:.4f}")

# Comparaison avant / après optimisation
compare_df = pd.DataFrame([
    {"Modèle": "RF base",       "RMSE": results["RandomForest"]["rmse"],  "R²": results["RandomForest"]["r2"]},
    {"Modèle": "RF tuned",      "RMSE": rf_tuned_res["rmse"],             "R²": rf_tuned_res["r2"]},
    {"Modèle": "XGBoost base",  "RMSE": results["XGBoost"]["rmse"],       "R²": results["XGBoost"]["r2"]},
    {"Modèle": "XGBoost tuned", "RMSE": xgb_tuned_res["rmse"],            "R²": xgb_tuned_res["r2"]},
])
print("\n=== Avant / Après optimisation ===")
print(compare_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Impact de l'optimisation", fontsize=13, fontweight="bold")
for ax, metric in zip(axes, ["RMSE", "R²"]):
    bars = ax.bar(compare_df["Modèle"], compare_df[metric],
                  color=["#aec6cf", "#2196F3", "#ffb3b3", "#F44336"], edgecolor="black")
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=30)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h * 1.01,
                f"{h:,.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# CELLULE 9 — IMPORTANCE DES FEATURES

importances = pd.Series(
    final_model.feature_importances_, index=feature_cols
).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
colors_imp = ["tomato" if v == importances.max() else "steelblue" for v in importances]
importances.plot(kind="barh", color=colors_imp[::-1], edgecolor="black")
plt.title(f"Importance des features — {final_res['name']}", fontweight="bold")
plt.xlabel("Importance relative")
plt.axvline(importances.mean(), color="orange", linestyle="--", label="Moyenne")
plt.legend()
plt.tight_layout()
plt.show()

print("Importance des features :")
print(importances.sort_values(ascending=False).to_string())


In [ ]:
# CELLULE 10 — ANALYSE DES RÉSIDUS

residuals = y_test.values - final_res["pred"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Analyse des résidus — {final_res['name']}", fontsize=13, fontweight="bold")

axes[0].hist(residuals, bins=30, color="steelblue", edgecolor="black")
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title("Distribution des résidus")
axes[0].set_xlabel("Résidu (réel - prédit)")
axes[0].set_ylabel("Fréquence")

axes[1].scatter(final_res["pred"], residuals, alpha=0.5, color="steelblue")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Résidus vs Valeurs prédites")
axes[1].set_xlabel("Prix prédit")
axes[1].set_ylabel("Résidu")

plt.tight_layout()
plt.show()

In [ ]:
import pickle
from snowflake.ml.registry import Registry

#  1. Sauvegarde pickle dans le Stage 
with open("/tmp/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open("/tmp/best_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

session.sql("CREATE STAGE IF NOT EXISTS model_stage").collect()
session.file.put("/tmp/scaler.pkl",     "@model_stage", overwrite=True)
session.file.put("/tmp/best_model.pkl", "@model_stage", overwrite=True)
print("Fichiers pickle uploadés dans @model_stage ")

#  2. Enregistrement dans le Snowflake Model Registry 
import pandas as pd
reg = Registry(
    session=session,
    database_name="HOUSE_PRICE_DB",
    schema_name="ML_SCHEMA"
)

model_version = reg.log_model(
    model=final_model,
    model_name="HOUSE_PRICE_MODEL",
    version_name="V1",
    sample_input_data=X_test_scaled[:5],
    comment=f"Meilleur modèle : {final_res['name']}",
    metrics={
        "r2"  : round(float(final_res['r2']),   4),
        "rmse": round(float(final_res['rmse']),  2),
        "mae" : round(float(final_res['mae']),   2),
    }
)
print("Modèle enregistré dans le Model Registry")

print(f"""
  Modèle    : {final_res['name']}
  R²        : {final_res['r2']:.4f}
  RMSE      : {final_res['rmse']:,.0f}
  MAE       : {final_res['mae']:,.0f}
  Registry  : HOUSE_PRICE_DB.ML_SCHEMA → HOUSE_PRICE_MODEL (V1)
  Stage     : @HOUSE_PRICE_DB.ML_SCHEMA.model_stage
""")
